## Legal Judgment Summarizer (InLegalBERT + Logistic Regression)

## The whole idea
1. **InLegalBERT** reads a sentence and turns it into 768 numbers (called an *embedding*). Similar sentences get similar numbers.
2. We already know which training sentences belong in the summary (label 1) and which do not (label 0), from `label.py`.
3. A **logistic regression** classifier learns the pattern: "numbers like this = summary sentence".
4. For a new judgment:- score every sentence, keep the top ones, put them back in the original order. That is the summary.

```
sentence --> InLegalBERT --> 768 numbers --> Logistic Regression --> score (0 to 1)
```

In [1]:
!pip install transformers

In [2]:
!pip install torch

In [3]:
!pip install -q rouge-score

In [1]:
import os, time
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

from huggingface_hub import snapshot_download

for attempt in range(1, 8):
    try:
        path = snapshot_download(
            "law-ai/InLegalBERT",
            local_dir="inlegalbert_local",
            allow_patterns=["config.json", "pytorch_model.bin", "tokenizer_config.json",
                               "vocab.txt", "special_tokens_map.json"],
           )
        print("Downloaded to:", path)
        break
    except Exception as e:
        print(f"Attempt {attempt} failed: {type(e).__name__}. Retrying in 10 seconds...")
        time.sleep(10)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Downloaded to: D:\Legal Case Text Summarization\inlegalbert_local


In [2]:
import json, glob, os
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from rouge_score import rouge_scorer
import joblib

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

Running on: cpu


## Step 2: Settings and data
- `N_TRAIN_DOCS` is how many training judgments to use.
- `RATIO` is the share of sentences kept in the summary. About 17% of sentences were labelled 1, so we use 0.17.


In [6]:
N_TRAIN_DOCS = 100
RATIO = 0.17

def find(split):
    hits = glob.glob(f"D:\Legal Case Text Summarization\processed\\{split}\labeled.jsonl", recursive=True)
    return hits[0] if hits else f"processed/{split}/labeled.jsonl"

def read_jsonl(path, n=None):
    docs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            docs.append(json.loads(line))
            if n and len(docs) >= n:
                break
    return docs

train_docs = read_jsonl(find("train"), N_TRAIN_DOCS)
test_docs = read_jsonl(find("test"))
test_docs = test_docs[:20]
print("training judgments:", len(train_docs), "| test judgments:", len(test_docs))

d = train_docs[0]
print("First judgment has", len(d["sentences"]), "sentences,", sum(d["labels"]), "labelled 1")

training judgments: 100 | test judgments: 20
First judgment has 89 sentences, 2 labelled 1


## Step 3: Load InLegalBERT and make the "sentence to numbers" function
- The **tokenizer** cuts a sentence into small pieces (tokens) that BERT understands.
- **BERT** gives one vector per token. We take the **average** over the tokens of the sentence, which gives one vector of 768 numbers per sentence.
- Sentences longer than 128 tokens are cut. That is enough for almost every legal sentence.

In [5]:
MODEL_NAME = "inlegalbert_local"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

@torch.no_grad()
def embed(sentences, batch_size=64):
    """Turn a list of sentences into an array of shape (number of sentences, 768)."""
    out = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=128,
                        return_tensors="pt").to(device)
        hidden = bert(**enc).last_hidden_state                 # one vector per token
        mask = enc["attention_mask"].unsqueeze(-1)             # ignore padding tokens
        mean = (hidden * mask).sum(1) / mask.sum(1)            # average = sentence vector
        out.append(mean.cpu().numpy())
    return np.vstack(out)

# quick check
vec = embed(["The appeal is allowed."])
print("One sentence becomes", vec.shape[1], "numbers")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: inlegalbert_local
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


One sentence becomes 768 numbers


## Step 4: Turning all training sentences into numbers

In [7]:
train_sents = [s for d in train_docs for s in d["sentences"]]
train_labels = np.array([l for d in train_docs for l in d["labels"]])

X_train = embed(train_sents)
print("X_train shape:", X_train.shape, "(sentences x numbers)")
print("Share labelled 1: {:.1f}%".format(100 * train_labels.mean()))

KeyboardInterrupt: 

## Step 5: Training the classifier

In [ ]:
clf = LogisticRegression(class_weight="balanced", max_iter=1000)
clf.fit(X_train, train_labels)
print("Training accuracy:", round(clf.score(X_train, train_labels), 3))

## Step 6: Make a summary
For one judgment:
1. Turn every sentence into numbers.
2. Ask the classifier for a score (a probability from 0 to 1) for each sentence.
3. Keep the top `RATIO` share of sentences with the highest scores.
4. Sort the kept sentences back into their **original order**, so the summary reads naturally.

In [ ]:
def summarize(sentences, ratio=RATIO):
    scores = clf.predict_proba(embed(sentences))[:, 1]        # probability of label 1
    k = max(1, round(len(sentences) * ratio))                  # how many to keep
    top = np.argsort(scores)[::-1][:k]                         # positions of the best k sentences
    return [sentences[i] for i in sorted(top)]                 # restore original order

def lead_baseline(sentences, ratio=RATIO):
    """Simple comparison: just take the first sentences."""
    k = max(1, round(len(sentences) * ratio))
    return sentences[:k]

demo = test_docs[0]
summary = summarize(demo["sentences"])
print("Judgment sentences:", len(demo["sentences"]), "| summary sentences:", len(summary))
print()
for s in summary[:6]:
    print("-", s)

## Step 7: Measure quality with ROUGE
ROUGE compares our summary with the human-written headnote by counting shared words.
- **ROUGE-1**: shared single words. **ROUGE-2**: shared word pairs. **ROUGE-L**: longest shared word sequence.
- Higher is better.

We compare against the **Lead baseline** (first sentences only). If our model beats it, the model has really learned something.

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def average_rouge(summaries, references):
    totals = {"rouge1": 0, "rouge2": 0, "rougeL": 0}
    for pred, ref in zip(summaries, references):
        result = scorer.score(ref, pred)                       # (reference, prediction)
        for name in totals:
            totals[name] += result[name].fmeasure
    return {name: round(100 * v / len(summaries), 2) for name, v in totals.items()}

references = [" ".join(d["summary_sentences"]) for d in test_docs]
ours  = [" ".join(summarize(d["sentences"])) for d in test_docs]
lead  = [" ".join(lead_baseline(d["sentences"])) for d in test_docs]

print("Lead baseline  :", average_rouge(lead, references))
print("Our model      :", average_rouge(ours, references))

## Step 8: Save the model
The web app will load these two things: InLegalBERT (downloaded by name) and the small classifier file.

In [ ]:
joblib.dump(clf, "classifier.joblib")
print("Saved classifier.joblib  (download it from the Output tab on Kaggle)")

## How to explain this to someone (30 seconds)
"Court judgments are long. I split each one into sentences and used InLegalBERT, a BERT model pretrained on Indian legal text, to convert each sentence into numbers. I labelled which sentences matched the official headnote, then trained a logistic regression to predict, for any sentence, whether it belongs in a summary. To summarize a new judgment, the model scores every sentence, keeps the top 17%, and shows them in their original order. I evaluated it with ROUGE against the human headnotes and compared it with a simple baseline."